In [1]:
import os

STORE_DIR = "./rag_store"
CHROMA_DIR = os.path.join(STORE_DIR, "chroma")
PARENTS_PATH = os.path.join(STORE_DIR, "parents.jsonl")

print("STORE_DIR exists:", os.path.isdir(STORE_DIR))
print("CHROMA_DIR exists:", os.path.isdir(CHROMA_DIR))
print("parents.jsonl exists:", os.path.isfile(PARENTS_PATH))

if os.path.isfile(PARENTS_PATH):
    # show first line to sanity-check
    with open(PARENTS_PATH, "r", encoding="utf-8") as f:
        print("parents.jsonl first line preview:")
        print(f.readline()[:300], "...")

STORE_DIR exists: True
CHROMA_DIR exists: True
parents.jsonl exists: True
parents.jsonl first line preview:
{"parent_id": "item_1000", "fingerprint": "f872aa2792c603beb2b1e3ef9b2a7c499e1f19d2", "text": "April 8, 1964\n\nDr. Horst Flegel\n4 Dusseldorf\nBayerische Landstrasse 2\nWest Germany\n\nDear Dr. Flegel:\n\nWe have received a letter from Dr. Boger, and it is only necessary that we receive a letter fr ...


In [1]:
import os
import sys
from pathlib import Path

# Ensure project root and tests/ are on sys.path
ROOT = Path.cwd()  # assuming you run the notebook from project root
sys.path.append(str(ROOT / "tests"))

from RAGSystem import RAGSystem

STORE_DIR = "./rag_store"

rag = RAGSystem(
    store_dir=STORE_DIR,
    chroma_collection="rag_collection",
    enable_bm25=True,          # can set False to simplify
    k_recall=30,
    k_ensemble=20,
    k_after_rerank=6,
)

print("Loaded parents:", len(rag._parent_lookup))
print("BM25 enabled:", rag.bm25 is not None)

/Users/nijjohnson/Projects/max-fink-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-01-12 14:30:40,747 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-01-12 14:30:40,748 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


/Users/nijjohnson/Projects/max-fink-rag/tests/RAGSystem.py:105: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)


2026-01-12 14:30:41,663 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


/Users/nijjohnson/Projects/max-fink-rag/tests/RAGSystem.py:106: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vs = Chroma(


2026-01-12 14:31:00,485 - sentence_transformers.cross_encoder.CrossEncoder - INFO - Use pytorch device: mps
Loaded parents: 6529
BM25 enabled: True


In [2]:
query = "Maximilian"

# retriever from Chroma
vec_ret = rag.vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# use invoke() instead of get_relevant_documents()
hits = vec_ret.invoke(query)

print("Child hits:", len(hits))
for i, d in enumerate(hits[:5], 1):
    md = d.metadata or {}
    print(f"\n[{i}] parent_id={md.get('parent_id')} title={md.get('Title') or md.get('title')}")
    print("source:", md.get("item_url") or md.get("Source") or md.get("source"))
    print("collection:", md.get("collection"))
    print(d.page_content[:300], "...")

Child hits: 5

[1] parent_id=item_6714 title=Correspondence to: Wyatt, Richard J.
source: https://exhibits.library.stonybrook.edu/mfp/items/show/6714
collection: Correspondence
Max ...

[2] parent_id=item_4473 title=Correspondence to: Itil, Turan M.
source: https://exhibits.library.stonybrook.edu/mfp/items/show/4473
collection: Correspondence
Thanks.

Max ...

[3] parent_id=item_6701 title=Correspondence to: Taylor, Michael A.
source: https://exhibits.library.stonybrook.edu/mfp/items/show/6701
collection: Correspondence
All my best.

Max ...

[4] parent_id=item_235 title=Perception of simultaneous tactile stimuli in normal children. Neurology. 1953 Jan; 3(1): 27-34. And, Spinal Fluid Findings Following Cerebral Angiography.
source: https://exhibits.library.stonybrook.edu/mfp/items/show/235
collection: Published Works
Maximilian Fink, M.D. ...

[5] parent_id=item_4421 title=Correspondence to: Itil, Turan M.
source: https://exhibits.library.stonybrook.edu/mfp/items/show/4421
collection: 

In [3]:
# Grab one child hit and locate its parent
child = hits[0]
pid = child.metadata.get("parent_id")

parent = rag._parent_lookup.get(pid)
print("Parent found:", parent is not None)
if parent:
    print("Parent title:", parent.metadata.get("Title") or parent.metadata.get("title"))
    print("Parent url:", parent.metadata.get("item_url"))
    print("\nParent text preview:\n", parent.page_content[:600], "...")

Parent found: True
Parent title: Correspondence to: Wyatt, Richard J.
Parent url: https://exhibits.library.stonybrook.edu/mfp/items/show/6714

Parent text preview:
 To: "Wyatt, Richard J." <WYATTR@dirpc.nimh.nih.gov>
Subject: Re: Your advice is needed.
Date sent: Tue, 13 May 1997 20:32:01

Dear Richard,

It is difficult to answer the generic questions about a patient in un-named country. I would hesitate to recommend ECT for anyone I cared for in some countries, and would not hesitate at all in others. So -- what country are we talking about?

Specifics -- as best as I can answer:

1. ECT is an appropriate treatment for paranoid and/or delusional psychoses of any etiology, especially if they have failed to respond to available treatments.

The ability to  ...


In [4]:
question = "What does Max Fink ask Dr. Horst Flegel to clarify in the letter?"
answer = rag.ask(question, chat_session_id="nb_test")
print(answer)

2026-01-12 14:32:31,400 - langchain_classic.retrievers.multi_query - INFO - Generated queries: ['Here are three alternative versions of the original question, each with a different perspective:', 'What specific information or clarification is requested from Dr. Horst Flegel by Max Fink in their written correspondence?', 'In what context does Max Fink ask for additional information or explanation about something related to his research or work from Dr. Horst Flegel through letter?', 'What particular point of inquiry, question, or topic of discussion is initiated by Max Fink and directed towards Dr. Horst Flegel in the exchanged correspondence.', 'These alternative questions aim to capture different aspects of the original query, encouraging the vector database to retrieve documents that might contain relevant information from various angles.']


Batches: 100%|██████████| 6/6 [00:04<00:00,  1.46it/s]


I don't know what Max Fink asks Dr. Horst Flegel to clarify in the letter on April 8, 1964. However, I can tell you that in the previous letters exchanged between them (February 17 and March 7, 1964), Max Fink was trying to clarify some points with Dr. Flegel, including the duration of time he planned to spend in the United States and the possibility of working as a laboratory scientist instead of a clinical researcher due to his lack of an ECMG examination certificate.
